In [1]:
%cd ..
import os
from pathlib import Path
from tqdm import tqdm


import cv2
import pandas as pd

from engine.utils.metrics.metric import (
    MAEmeasure,
    Smeasure,
    Emeasure,
    Fmeasure,
    WeightedFmeasure,
)

os.chdir("/Users/felix/Desktop/UCOD-DPL")

gt_dir = Path("datasets/RefCOD/TE-COD10K/gt")
pred_dir = Path("results/preds/TE-COD10K")

rows = []

for gt_path in tqdm(sorted(gt_dir.glob("*")), desc="Evaluating images"):
    if gt_path.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
        continue

    pred_path = pred_dir / f"{gt_path.stem}.png"
    if not pred_path.exists():
        pred_path = pred_dir / f"{gt_path.stem}.jpg"

    if not pred_path.exists():
        rows.append({"image": gt_path.name, "error": "missing prediction"})
        continue
    gt = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
    pred = cv2.imread(str(pred_path), cv2.IMREAD_GRAYSCALE)

    if gt is None or pred is None:
        rows.append({"image": gt_path.name, "error": "read failed"})
        continue

    pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]))

    mae = MAEmeasure()
    sm = Smeasure()
    em = Emeasure()
    fm = Fmeasure()
    wfm = WeightedFmeasure()

    em.step(pred=pred, gt=gt)
    sm.step(pred=pred, gt=gt)
    fm.step(pred=pred, gt=gt)
    mae.step(pred=pred, gt=gt)
    wfm.step(pred=pred, gt=gt)

    em_result = em.get_results()["em"]
    fm_result = fm.get_results()["fm"]

    rows.append({
        "image": gt_path.name,
        "MAE": mae.get_results()["mae"],
        "SMeasure": sm.get_results()["sm"],
        "E_MAX": em_result["curve"].max(),
        "E_MEAN": em_result["curve"].mean(),
        "F_MAX": fm_result["curve"].max(),
        "F_MEAN": fm_result["curve"].mean(),
        "WFM": wfm.get_results()["wfm"],
        "gt_path": str(gt_path),
        "pred_path": str(pred_path),
    })

df = pd.DataFrame(rows)
df.head()

/Users/felix/Desktop/UCOD-DPL


Evaluating images: 100%|██████████| 2026/2026 [03:34<00:00,  9.44it/s]


,image,MAE,SMeasure,E_MAX,E_MEAN,F_MAX,F_MEAN,WFM,gt_path,pred_path
0,COD10K-CAM-1-Aquatic-1-BatFish-2.png,0.034550,0.902834,0.966820,0.964020,0.929577,0.926976,0.912648,datasets/RefCOD/TE-COD10K/gt/COD10K-CAM-1-Aqua...,results/preds/TE-COD10K/COD10K-CAM-1-Aquatic-1...
1,COD10K-CAM-1-Aquatic-1-BatFish-4.png,0.082607,0.836815,0.914219,0.911624,0.889226,0.887211,0.849687,datasets/RefCOD/TE-COD10K/gt/COD10K-CAM-1-Aqua...,results/preds/TE-COD10K/COD10K-CAM-1-Aquatic-1...
2,COD10K-CAM-1-Aquatic-1-BatFish-5.png,0.025056,0.925910,0.976503,0.973665,0.934938,0.932262,0.925749,datasets/RefCOD/TE-COD10K/gt/COD10K-CAM-1-Aqua...,results/preds/TE-COD10K/COD10K-CAM-1-Aquatic-1...
3,COD10K-CAM-1-Aquatic-1-BatFish-6.png,0.014235,0.874564,0.977516,0.974674,0.790430,0.787529,0.785452,datasets/RefCOD/TE-COD10K/gt/COD10K-CAM-1-Aqua...,results/preds/TE-COD10K/COD10K-CAM-1-Aquatic-1...
4,COD10K-CAM-1-Aquatic-10-LeafySeaDragon-416.png,0.041651,0.799412,0.953670,0.950922,0.700132,0.697751,0.680878,datasets/RefCOD/TE-COD10K/gt/COD10K-CAM-1-Aqua...,results/preds/TE-COD10K/COD10K-CAM-1-Aquatic-1...


In [2]:
from pathlib import Path

save_dir = Path("results/metric_analysis")
save_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(save_dir / "te-cod10k-per-image-metrics.csv", index=False)